Fazendo a minha primeira chamada da LLM 

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv 
import os

load_dotenv()

/Users/brunavitor/Downloads/ASSISTENTE_IA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Criando a primeira chamada de LLM

In [2]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GOOGLE_API_KEY"))  

In [3]:
llm.invoke("Quem foi Maria Antonieta?").content

'**Maria Antonieta (originalmente Marie Antoinette Josephe Jeanne d\'Autriche-Lorraine)** foi a última rainha da França antes da Revolução Francesa. Ela era a esposa do rei Luís XVI e se tornou um dos símbolos mais proeminentes da monarquia absolutista e do luxo da corte de Versalhes, bem como uma figura central na tragédia que culminou na queda da realeza francesa.\n\nAqui estão os pontos chave sobre quem ela foi:\n\n1.  **Origem e Casamento Político:**\n    *   Nascida em 1755 em Viena, Áustria, ela era a filha mais nova da Imperatriz Maria Teresa da Áustria e de Francisco I do Sacro Império Romano-Germânico.\n    *   Aos 14 anos, em 1770, casou-se com o Delfim da França (futuro Luís XVI) como parte de uma aliança política entre a Casa de Habsburgo (Áustria) e a Casa de Bourbon (França).\n\n2.  **Rainha da França:**\n    *   Tornou-se rainha em 1774, quando Luís XVI ascendeu ao trono.\n    *   Na corte francesa, sua personalidade vibrante e seu gosto por festas, moda extravagante, jo

Construindo a primeira estrutura de RAG

In [4]:
import sys
from pathlib import Path
import pandas as pd


# Importando da community, que é mais flexível com as versões do seu Mac
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Configurações de caminho
BASE_DIR = Path().resolve()
DOCS_DIR = BASE_DIR / "docs"



Escolhendo a estrutura de embedding

In [5]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/y6/zsd4qv5n4h146wxhv01rxltc0000gn/T/ipykernel_2656/323160131.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [6]:

def load_pdf_vectorstore(filepath: str, save_path: str):
    loader = PyPDFLoader(DOCS_DIR / filepath)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=500)
    documents = text_splitter.split_documents(documents)
    vectorstore = FAISS.from_documents(documents, embedding)    
    vectorstore.save_local(f'vectostores/{save_path}')
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 7}) 
    return retriever

In [7]:
retriever_perguntas_frequentes = load_pdf_vectorstore("Perguntas Frequentes.pdf", "vectorstore_perguntas_frequentes")
retriever_manual_tecnico = load_pdf_vectorstore("Manual Tecnico de Produtos.pdf", "vectorstore_manual_tecnico_produtos")
retriever_politicas_procedimentos = load_pdf_vectorstore("Politicas e Procedimentos.pdf", "vectorstore_politica_procedimentos")   

In [8]:
def load_excel_vectorstore(filepath: str, save_path: str):
    df =  pd.read_excel(DOCS_DIR / filepath)
    documents = []
    for idx, row in df.iterrows():
        text  = ' '.join([str(cell) for cell in row if pd.notna(cell)])
        documents.append(Document(page_content=text, metadata={'row': idx}))
      
    vectorstore_tickets = FAISS.from_documents(documents, embedding)    
    vectorstore_tickets.save_local('vectostores/vectorstore_tickets')
    retriever_tickets = vectorstore_tickets.as_retriever(search_type="similarity", search_kwargs={"k": 7})
    
    return retriever_tickets

In [9]:
retriever_tickets = load_excel_vectorstore("Tickets.xlsx", "vectorstore_tickets")

Criando os agentes assistentes

In [10]:
#importando bibliotecas
from typing import TypedDict, Optional , List #tipando os dados
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage #definindo o formato da mensagem

Estrutura de mensagens e padrão que está sendo percorrido por detrás dos panos

In [11]:
class State(TypedDict, total=False): #herdando de TypedDict e total=False para permitir chaves opcionais
  query: str
  route : Optional[str] #rota é opcional, pode ser str ou None
  anwser: Optional[str] #resposta é opcional, pode ser str ou None
  chat_history: Optional[List[BaseMessage]] #histórico de mensagens é opcional, pode ser uma lista de mensagens ou None   
  

In [12]:
def agent_with_retriever(state: State, papel: str, prompt_instructions: str, retriever = None):
    query = state["query"] #recuperar pergunta feita pelo usuário
    chat_history = state.get("chat_history", []) #recuperar histórico de mensagens, se não existir, usar lista vazia
    context = "" #inicializar contexto vazio
   
    if retriever:
        recuperados = retriever.get_relevant_documents(query) #recuperar documentos relevantes usando o retriever
        if recuperados: #se houver documentos recuperados, construir o contexto concatenando o conteúdo das páginas
            context = "\n".join([doc.page_content for doc in recuperados]) #construir o contexto concatenando o conteúdo das páginas recuperadas, separando por nova linha

  #criando estrutura de mensagens. A primeira mensagem é do sistema, definindo o papel do agente e as instruções. Em seguida, adicionamos o histórico de mensagens da conversa, se houver, e por fim, a pergunta atual do usuário junto com o contexto recuperado.
    mensagens = [
        SystemMessage(content=
                      f"Você é um {papel}."
                      f"Suas instruoes:\n{prompt_instructions}\n\n"
                      f" - Use sempre o contexto recuperado para responder a última pergunta do usuário.\n"
                      f" - Use o histórico da conversa para entender o contexto geral e perguntas de acompanhamento. \n"
                      f" - Se nao houver informaçoes relevantes no contexto, diga que nao encontrou dados suficientes para responder a pergunta. \n"
                      f" - Evite inventar informações."
                 ),
                *chat_history, #desempacotar o histórico de mensagens e adicioná-lo à lista de mensagens
          HumanMessage(content=(
               f"Pergunta do usuário: \n{query}\n\n"
               f"Contexto disponivel para esta pergunta: \n{context if context else 'Nenhum contexto disponível.'}"
           )) #adicionar a pergunta atual do usuário como uma mensagem do tipo
    ]
    resposta = llm.invoke(mensagens)#gerar resposta usando o modelo de linguagem, passando a lista de mensagens como entrada
    state["anwser"] = resposta.content #atualizar o estado com a resposta gerada
    return state

Criando o agente de detalhe técnico

In [13]:
def agent_detalhe_tecnico(state: State):
    prompt_instructions = (
        "Seja um **especialista em suporte técnico e produto**. " 
        "Você deve responder a perguntas sobre **especificações técnicas**."
        "**instrucoes de instalação**, **manutençao preventiva** e **soluçao de problemas**."
        "Sua resposta deve ser precisa, técnica e objetiva, baseada estritamente no manualk técnico."
        "Para problemas, ofereça uma soluçao clara e passo a passo."
    )
    return agent_with_retriever(
        state, "especialista em detalhes técnicos de produtos", prompt_instructions, retriever_manual_tecnico    )

Criando o agente de perguntas frequentes

In [14]:
def agent_perguntas_e_respostas(state: State):
    prompt_instructions = (
        "Seja um **especialista em Perguntas e Respostas (FAQ)**. " 
        "Sua função é fornecer respostas diretas e concisas a perguntas comuns."
        "Responda como se estivesse consultando uma base de conhecimento, mantendo a resposta factual e sem rodeios."
        "Se a pergunta se referir a um problema, ofereça a resposta e, se necessário, sugira o contato com o suporte técnico para casos complexos."
    )
    return agent_with_retriever(
        state, "especialista em FAQs", prompt_instructions, retriever_perguntas_frequentes    )

criando o agente de politicas e procedimentos

In [15]:
def agent_politicas_e_procedimentos(state: State):
    prompt_instructions = (
        "Seja um **especialista em políticas e procedimentos da empresa**. " 
        "Sua tarefa é responder a perguntas sobre **garantia** e **horário de atendimento**."
        "**prazos de SLA** e **regras internas de suporte**."
        "Sua resposta deve ser formal e baseada nos documentos oficiais, farantindo que o cliente entenda as regras e os processos da empresa."
    )
    return agent_with_retriever(
        state, "especialista em políticas e procedimentos da empresa", prompt_instructions, retriever_politicas_procedimentos     )

Criando o agente de tickets

In [16]:
def agent_tickets(state: State):
    prompt_instructions = (
        "Seja um **especialista em tickets de atendimento**. " 
        "Você deve fornecer informações precisas sobre **status e detalhes de um chamado existente**."
        "Sua resposta deve ser direta, baseada nos dados do ticket (ticket ID, Status, Responsável, Descrição do produto)."
        "Se o usuário perguntar sobre um ticket específico, forneça as informações correspondentes e relevantes e mantenha a resposta curta e direta."
    )
    return agent_with_retriever(
        state, "especialista em tickets de atendimento", prompt_instructions, retriever_tickets    )

Criando Agente Supervisor

In [17]:
def agent_supervisor(state: State):
    query = state["query"]
    chat_history = state.get("chat_history", [])

    mensagens = [SystemMessage(content=(
        """Você é um assistente virtual de atendimento ao cliente da **Industech**, uma empresa especializada em produtos industriais.
        Sua principal responsabilidade é atuar como supervisor de atendimento ao cliente, **roteando as perguntas dos clientes para o agente especialista mais adequado**.
        Sua única função é analisar a pergunta do usuário e retornar **uma palavra-chave** que representa o agente responsável. Se a pergunta não se encaixar em nenhuma das categorias de especialistas, ou se for uma saudação ou uma pergunta sobre suas próprias capacidades, você deve responder de forma amigável ao cliente, sem rotear.

        A sua resposta deve ser:
        -**Uma frase de resposta direta**, caso a pergunta seja geral (ex: "Olá", "Tudo bem?", "O que você faz?").
        -**Uma das seguintes palavras-chave**, em minúsculas e sem pontuação, para roteamento:
        -**detalhe_tecnico**: para perguntas sobre **especificações técnicas**, **manuais de produtos**, ou **solução de problemas**.
        -**perguntas_e_respostas**: para **dúvidas operacionais comuns** ou **FAQs**.
        -**politicas_e_procedimentos**: para questões sobre **políticas da empresa**, **garantia** ou **prazos de atendimento (SLA)**.
        -**tickets**: para perguntas sobre o **status de um chamado existente** ou **detalhes de um ticket**.

        **Regras:**
        -Responda apenas com a frase ou com a palavra-chave.
        -Não adicione explicações, comentários ou qualquer outro texto.
        -não invente novas categorias."""
    )),
    *chat_history,
    HumanMessage(content=query)]
    resposta = llm.invoke(mensagens)
    resposta_limpa = resposta.content.strip().lower() #limpar a resposta removendo espaços em branco e convertendo para minúsculas
    if resposta_limpa in ["detalhe_tecnico", "perguntas_e_respostas", "politicas_e_procedimentos", "tickets"]:
        state["route"] = resposta_limpa #atualizar o estado com a rota correspondente à palavra-chave identificada
    else:
        state["anwser"] = resposta.content #se a resposta não for uma palavra-chave de roteamento, atualizar o estado com a resposta gerada para perguntas gerais
    return state

Workflow

In [18]:
from langgraph.graph import StateGraph, START, END

In [19]:
#funçao para decidir o proximo passo
def decide_action(state: State):
    #se a resposta já foi gerada pelo supervisor, encerra o fluxo.
    if 'answer' in state:
        return 'end_workflow'
    else:
        #caso contrario, usa a rota para ir para o agente especialista correspondente
        return state['route']

In [23]:
def build_workflow():
    workflow = StateGraph(State)

    #adicionar os nós
    workflow.add_node('supervisor_node', agent_supervisor) 
    workflow.add_node('detalhe_tecnico_node', agent_detalhe_tecnico) 
    workflow.add_node('perguntas_e_respostas_node', agent_perguntas_e_respostas) 
    workflow.add_node('politicas_e_procedimentos_node', agent_politicas_e_procedimentos) 
    workflow.add_node('tickets_node', agent_tickets) 

    #adicionar um nó de saída para a resposta direta
    workflow.add_node('end_workflow', lambda x: x)

    #definir o nó inicial
    workflow.add_edge(START, 'supervisor_node')

    #adicione o roteamento condicional
    workflow.add_conditional_edges(
                                  'supervisor_node',
                                  decide_action,
                                  {
                                      "detalhe_tecnico": 'detalhe_tecnico_node',
                                      "perguntas_e_respostas": 'perguntas_e_respostas_node',
                                      "politicas_e_procedimentos": 'politicas_e_procedimentos_node',
                                      "tickets": 'tickets_node',
                                      "end_workflow": END #determina o fluxo se a resposta já foi gerada pelo supervisor.
                                  }                                
 )
     #defina as saídas dos agentes especialistas
    workflow.add_edge('detalhe_tecnico_node', END)
    workflow.add_edge('perguntas_e_respostas_node', END)
    workflow.add_edge('politicas_e_procedimentos_node', END)
    workflow.add_edge('tickets_node', END)
    
    return workflow.compile()

In [25]:
app = build_workflow()